[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/08_%E8%A8%82%E9%96%B1_%E6%9C%83%E5%93%A1%E6%B5%81%E5%A4%B1%E6%8F%90%E6%97%A9%E6%8C%BD%E5%9B%9E/%E6%9C%83%E5%93%A1%E6%B5%81%E5%A4%B1%E6%8F%90%E6%97%A9%E6%8C%BD%E5%9B%9E_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb)

# 🔔 範例 08：訂閱營運—會員流失提早挽回 (不平衡分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「SaaS 軟體產品經理、數位訂閱營運人員與顧客成功經理（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以訂閱留存白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、SaaS 訂閱資料集介紹 (`churn.csv`)

本數據集記錄了 SaaS 訂閱服務 2,200 位訂戶的數位足跡與次月續訂成效（流失率僅約 9.3%）：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 訂閱業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **訂閱方案** | 類別型 (`object`) | 個人標準版、專業進階版、企業尊榮版 | 付費級別與功能門檻 |
| **已訂閱月數** | 數值型 (`int64`) | 3 (個月) | 留存月數（前 3 個月為流失高危期） |
| **最近30天登入天數** | 數值型 (`int64`) | 2 (天) | 活躍度核心指標（驟降為退訂第一前兆） |
| **上月使用時長_小時** | 數值型 (`float64`) | 1.5 (小時) | 深度黏著指標 |
| **最近客服抱怨次數** | 數值型 (`int64`) | 2 (次) | 遇挫求助次數 |
| **是否綁定自動扣款** | 類別型 (`object`) | 是、否 | 續約摩擦力（未綁定者流失率高達數倍） |
| **社群互動次數** | 數值型 (`int64`) | 0 (次) | 社群黏著度 |
| **次月是否流失 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：次月是否退訂（9.3% 不平衡少數類）** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視 9.3% 極度不平衡流失比例。
3. **特徵關聯性與視覺化**：繪製登入天數與客服抱怨對退訂率之影響圖。
4. **特徵工程與 Pipeline 建置**：分層切分 8:2 訓練/測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「普通隨機森林」與「加權隨機森林 (class_weight='balanced')」。
6. **測試集大考與主動挽回策略**：解讀 PR-AUC、Recall，輸出模型並模擬高危客戶即時預警。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入不平衡流失預警分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並確保中文字型正常顯示。

---
💡 **AI 提問小抄：**

1. 📌 **準確率陷阱 (Accuracy Paradox)**
   > 「在我們的 2,200 筆訂戶中，只有 9.3% 的人會流失。如果我寫一個完全沒腦袋的程式，對所有人都猜『不會流失』，準確率就高達 90.7%！請向老闆解釋為什麼這種高準確率完全是自欺欺人？」


---  
### 【步驟 1】：載入資料與不平衡結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 churn.csv 並檢視資料概況
df = pd.read_csv("churn.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 次月流失標籤分佈比例（注意嚴重不平衡）---")
print(df["次月是否流失"].value_counts(normalize=True))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  檢查 2,200 筆訂戶數據，確認流失率僅約 9.3%，呈現典型少數類別不平衡狀態。

---
💡 **AI 提問小抄：**

1. 📌 **不平衡資料處理策略**
   > 「面對 1:9 的不平衡標籤，業界常見的解決方案有：欠抽樣 (Undersampling)、過抽樣 (SMOTE)、演算法懲罰權重 (class_weight)。這三種方法在實務上的優缺點各是什麼？」


---  
### 【步驟 2】：活躍登入天數與客服抱怨之流失交叉圖表繪製

In [ ]:
# [步驟 2 程式碼] 繪製登入天數與是否自動扣款的流失率對比
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x="次月是否流失", y="最近30天登入天數", palette="Set1", ax=axes[0])
axes[0].set_title("次月是否流失 vs 最近30天登入天數")
axes[0].set_ylabel("最近30天登入天數")

auto_churn = df.groupby("是否綁定自動扣款")["次月是否流失"].apply(lambda s: (s == "是").mean())
sns.barplot(x=auto_churn.index, y=auto_churn.values * 100, palette="Reds_r", ax=axes[1])
axes[1].set_title("是否綁定自動扣款之退訂流失率 (%)")
axes[1].set_ylabel("流失率 (%)")

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  透過箱型圖與長條圖，清晰展示流失者登入天數明顯偏低，以及未綁定自動扣款用戶流失率高達數倍的現象。

---
💡 **AI 提問小抄：**

1. 📌 **付費摩擦力與黏著度**
   > 「圖表中未綁定自動扣款的用戶流失率超過 20%，而綁定者低於 5%。這對 SaaS 產品在註冊時強制要求綁卡提供免費試用期的商業策略有何啟發？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構 (分層抽樣)

In [ ]:
# [步驟 3 程式碼] 使用 stratify=y 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["次月是否流失"])
y = (df["次月是否流失"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["已訂閱月數", "最近30天登入天數", "上月使用時長_小時", "最近客服抱怨次數", "社群互動次數"]
cat_cols = ["訂閱方案", "是否綁定自動扣款"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集流失樣本數：{y_train.sum()} 筆，測試集流失樣本數：{y_test.sum()} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  特別加上 `stratify=y` 確保訓練集與測試集中的流失比例完全維持 9.3%，避免測試集運氣不好分不到任何一個流失案例。

---
💡 **AI 提問小抄：**

1. 📌 **分層抽樣在不平衡數據中的致命重要性**
   > 「如果不加 `stratify=y`，在隨機切分時剛好 400 筆測試集只分到 2 個流失用戶，會對後續的模型評估造成什麼災難？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (引入 class_weight)

In [ ]:
# [步驟 4 程式碼] 比對普通隨機森林 vs 加權隨機森林 (class_weight='balanced')
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_default = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

pipe_weighted = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42))
])

scores_def = cross_val_score(pipe_default, X_train, y_train, cv=5, scoring="roc_auc")
scores_wt = cross_val_score(pipe_weighted, X_train, y_train, cv=5, scoring="roc_auc")

print(f"預設隨機森林 5 折交叉驗證 ROC-AUC: {scores_def.mean():.4f} (±{scores_def.std():.4f})")
print(f"加權隨機森林 (balanced) 5 折交叉驗證 ROC-AUC: {scores_wt.mean():.4f} (±{scores_wt.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  加入 `class_weight='balanced'` 讓演算法對猜錯流失者給予 10 倍重的數學懲罰，逼迫模型認真尋找流失特徵。

---
💡 **AI 提問小抄：**

1. 📌 **class_weight 運作原理**
   > 「`class_weight='balanced'` 在底層是如何依據樣本頻率自動計算權重的？它跟手動把少數類別複製 10 倍（Oversampling）在訓練時間與記憶體上有何差異？」


---  
### 【步驟 5】：測試集大考與指標解讀 (拒絕 Accuracy, 看 PR-AUC & Recall)

In [ ]:
# [步驟 5 程式碼] 評估測試集之分類報告與混淆矩陣
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

pipe_weighted.fit(X_train, y_train)
y_pred = pipe_weighted.predict(X_test)
y_prob = pipe_weighted.predict_proba(X_test)[:, 1]

print("=== 測試集不平衡分類報告 ===")
print(classification_report(y_test, y_pred, target_names=["留存(0)", "流失(1)"]))

precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)
print(f"PR-AUC (Precision-Recall 曲線下面積): {pr_auc:.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=["預測留存", "預測流失"], yticklabels=["實際留存", "實際流失"])
plt.title("會員流失預警混淆矩陣 (注意流失召回數)")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  輸出分類報告與 PR-AUC，重點檢視「實際流失」的 41 位會員中有多少人被成功抓出（Recall）。

---
💡 **AI 提問小抄：**

1. 📌 **商業挽留成本計算**
   > 「抓出一個流失客戶並挽留成功，公司保住每年 $12,000 元訂閱費；誤發一個 7 折折扣碼給不會走的人，公司少賺 $3,600 元。請幫我建立一個期望值公式，算出最能替公司賺錢的機率閾值門檻？」


---  
### 【步驟 6】：高危訂戶即時推論 (Inference) 與主動挽回策略

In [ ]:
# [步驟 6 程式碼] 保存模型管線並模擬高危訂戶即時預警
joblib.dump(pipe_weighted, "model.joblib")
print("✅ 不平衡加權模型管線已成功保存至 model.joblib！")

# 模擬一位距離續約扣款只剩 7 天的受挫會員
member = pd.DataFrame([{
    "訂閱方案": "個人標準版",
    "已訂閱月數": 2,
    "最近30天登入天數": 1,
    "上月使用時長_小時": 0.8,
    "最近客服抱怨次數": 2,
    "是否綁定自動扣款": "否",
    "社群互動次數": 0
}])

loaded_model = joblib.load("model.joblib")
churn_prob = loaded_model.predict_proba(member)[0][1]
print(f"\n🔔 該訂戶次月流失機率預估：{churn_prob*100:.1f}%")
if churn_prob >= 0.5:
    print("🚨 極高流失警報！觸發 CSM 客服專人通話，並於會員 App 首頁派發 7 折續約挽留抵用金！")
else:
    print("🟢 正常健康訂戶，持續提供常規功能更新推播。")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  封裝模型並示範即時評估高危險用戶，觸發自動化挽留旅程。

---
💡 **AI 提問小抄：**

1. 📌 **自動化挽留旅程設計 (Marketing Automation)**
   > 「在現代 SaaS 工具（如 HubSpot 或 Braze）中，如何將這個模型的推論結果作為 Webhook 觸發器，自動在流失機率大於 60% 時自動派送電子郵件與簡訊？」
